In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, lower, trim, when, current_timestamp, to_date,
    year, month, quarter, dayofweek, date_format, lit
)
import time



# ==== CONFIG ====
storage_account = "ecommersestorage"
catalog_name = "ecommersedatabrick_7405618595285403"
silver_schema = "silver"
gold_schema = "gold"

# 1. KẾT NỐI ĐẾN STORAGE ACCOUNT
storage_key = "my-HashKey"
spark.conf.set(f"fs.azure.account.key.{storage_account}.dfs.core.windows.net", storage_key)

spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")


# ==== OPTIMIZE ====
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled", "true")

start_time = time.time()

# ==== UNITY CATALOG ====
spark.sql(f"USE CATALOG `{catalog_name}`")
spark.sql(f"USE SCHEMA `{gold_schema}`")




DataFrame[]

In [0]:

print("Đọc dữ liệu từ Silver...")

orders = spark.table(f"{catalog_name}.{silver_schema}.orders")
products = spark.table(f"{catalog_name}.{silver_schema}.products")
sellers = spark.table(f"{catalog_name}.{silver_schema}.sellers")
customers = spark.table(f"{catalog_name}.{silver_schema}.customers")
geolocation = spark.table(f"{catalog_name}.{silver_schema}.geolocation")
order_items = spark.table(f"{catalog_name}.{silver_schema}.order_items")
payments = spark.table(f"{catalog_name}.{silver_schema}.order_payments")
reviews = spark.table(f"{catalog_name}.{silver_schema}.order_reviews")

display(df_orders_silver.limit(5))


Đọc dữ liệu từ Silver...


order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,is_valid,processed_at
f373335aac9a659de916f7170b8bc07a,f06a94a401e52fb019c72f2e8bbf6a2f,shipped,2018-03-17T15:32:31Z,2018-03-17T15:48:40Z,2018-03-20T21:08:28Z,null,2018-04-13T00:00:00Z,null,false,2026-03-25T07:15:28.808676Z
118045506e1c1dda060171af43fe11b4,638c6674418fc58283a73c078bcb076f,delivered,2018-03-08T19:06:05Z,2018-03-09T19:08:26Z,2018-03-13T21:24:28Z,2018-04-11T12:53:50Z,2018-04-04T00:00:00Z,34,true,2026-03-25T07:15:28.808676Z
cc66dee6fbc18bb79903c3a2cc14ff52,19d3b3a2d4756af17603e2c35c7c2815,delivered,2018-04-12T14:37:29Z,2018-04-12T15:15:27Z,2018-04-16T16:23:53Z,2018-04-20T17:28:56Z,2018-05-07T00:00:00Z,8,true,2026-03-25T07:15:28.808676Z
f44cb69655f8e4d13e7aae7cdd3d3c25,eab62436056c6ce3853a17dd6892951a,delivered,2018-07-13T22:22:57Z,2018-07-13T22:35:20Z,2018-07-24T19:07:00Z,2018-07-25T14:03:41Z,2018-07-31T00:00:00Z,12,true,2026-03-25T07:15:28.808676Z
edcc6b79e8394346ba3ba21b00b4055e,08aea10c40f606e52597486db2b56a81,delivered,2018-04-29T16:03:47Z,2018-04-29T16:15:25Z,2018-05-02T08:25:00Z,2018-05-11T23:12:12Z,2018-05-25T00:00:00Z,12,true,2026-03-25T07:15:28.808676Z


In [0]:

# ==== DIMENSIONS ====

dim_customers = (
    df_customers_silver.select(
        col("customer_id"),
        col("customer_unique_id"),
        col("customer_city"),
        col("customer_state"),
        col("customer_zip_code_prefix"),
    )
    .dropDuplicates(["customer_id"])
    .withColumn("gold_updated_at", current_timestamp())
)

dim_sellers = (
    df_sellers_silver.select(
        col("seller_id"),
        col("seller_city"),
        col("seller_state"),
        col("seller_zip_code_prefix"),
    )
    .dropDuplicates(["seller_id"])
    .withColumn("gold_updated_at", current_timestamp())
)

dim_products = (
    df_products_silver.select(
        col("product_id"),
        col("product_category_name").alias("category"),
        (col("product_length_cm") * col("product_height_cm") * col("product_width_cm")).alias("volume_cm3"),
        col("product_weight_g").alias("weight_g"),
    )
    .dropDuplicates(["product_id"])
    .withColumn("gold_updated_at", current_timestamp())
)

dim_orders = (
    df_orders_silver.select(
        col("order_id"),
        col("customer_id"),
        col("order_status"),
        col("order_purchase_timestamp"),
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date"),
        col("delivery_days"),
        col("is_valid"),
    )
    .dropDuplicates(["order_id"])
    .withColumn("gold_updated_at", current_timestamp())
)

dim_payments = (
    df_order_payments_silver.select(
        col("order_id"),
        col("payment_sequential"),
        col("payment_type"),
        col("payment_installments"),
        col("payment_value"),
    )
    .withColumn("gold_updated_at", current_timestamp())
)

dim_reviews = (
    df_order_reviews_silver.select(
        col("review_id"),
        col("order_id"),
        col("review_score"),
        col("review_comment_title"),
        col("review_creation_date"),
    )
    .dropDuplicates(["review_id"])
    .withColumn("gold_updated_at", current_timestamp())
)

dim_date = (
    df_orders_silver
    .select(to_date(col("order_purchase_timestamp")).alias("date"))
    .where(col("date").isNotNull())
    .distinct()
    .withColumn("date_key", date_format(col("date"), "yyyyMMdd").cast("int"))
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("quarter", quarter(col("date")))
    .withColumn("day_of_week", dayofweek(col("date")))
    .withColumn("day_name", date_format(col("date"), "EEEE"))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn("gold_updated_at", current_timestamp())
)

# ==== FACT ====
fact_sales = (
    df_order_items_silver.alias("oi")
    .join(
        df_orders_silver.alias("o"),
        on="order_id",
        how="left",
    )
    .select(
        col("oi.order_id"),
        col("o.customer_id"),
        # Tạo khóa ngày để nối với dim_date
        date_format(col("o.order_purchase_timestamp"), "yyyyMMdd").cast("int").alias("order_date_key"), 
        col("oi.order_item_id"),
        col("oi.product_id"),
        col("oi.seller_id"),
        col("o.order_purchase_timestamp"),
        col("o.order_status"),
        col("oi.price"),
        col("oi.freight_value"),
        (col("oi.price") + col("oi.freight_value")).alias("total_amount"),
    )
    .withColumn("gold_updated_at", current_timestamp())
)

print("✅ Đã tạo xong các bảng Gold")


✅ Đã tạo xong các bảng Gold


In [0]:

def save_uc_table(df, table_name):
    full_name = f"{catalog_name}.{gold_schema}.{table_name}"
    
    (df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(full_name))
    
    print(f"✅ Saved: {full_name}")

# ==== WRITE ALL GOLD TABLES ====

gold_tables = {
    "fact_sales": fact_sales,
    "dim_customers": dim_customers,
    "dim_sellers": dim_sellers,
    "dim_products": dim_products,
    "dim_orders": dim_orders,
    "dim_payments": dim_payments,
    "dim_reviews": dim_reviews,
    "dim_date": dim_date,
}

for name, df in gold_tables.items():
    save_uc_table(df, name)


print("✅ Hoàn thành Gold layer trong Unity Catalog.")

✅ Saved: ecommersedatabrick_7405618595285403.gold.fact_sales
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_customers
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_sellers
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_products
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_orders
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_payments
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_reviews
✅ Saved: ecommersedatabrick_7405618595285403.gold.dim_date
✅ Hoàn thành Gold layer trong Unity Catalog.
Thời gian xử lý: 24.77 giây


In [ ]:
# Tính toán ngày mua hàng muộn nhất trong toàn bộ hệ thống để làm mốc tham chiếu
from pyspark.sql import functions as F

ctx_max_date = fact_sales.select(F.max("shipping_limit_date")).collect()[0][0] 

# Tiến hành gom nhóm theo khách hàng để tính R, F, M
df_rfm_base = fact_sales.join(dim_customers, "customer_id", "inner") \
                        .groupBy("customer_unique_id") \
                        .agg(
                            # Recency: Số ngày kể từ lần mua cuối
                            F.datediff(F.lit(ctx_max_date), F.max("shipping_limit_date")).alias("recency"),
                            # Frequency: Tổng số đơn hàng (đếm số order_id duy nhất)
                            F.countDistinct("order_id").alias("frequency"),
                            # Monetary: Tổng tiền (giá sản phẩm + phí vận chuyển)
                            F.sum(F.col("price") + F.col("freight_value")).alias("monetary")
                        )

# Phân khúc khách hàng (Customer Segmentation) dựa trên các chỉ số RFM
df_crm_final = df_rfm_base.withColumn(
    "customer_segment",
    F.when((F.col("recency") <= 30) & (F.col("frequency") >= 3), "VIP / Loyal Customers")
     .when((F.col("recency") > 180), "Churned Customers (Rời bỏ)")
     .when((F.col("monetary") >= 500) & (F.col("recency") <= 90), "Big Spenders (Chi tiêu lớn)")
     .when((F.col("recency") <= 60) & (F.col("frequency") == 1), "New Customers (Khách mới)")
     .otherwise("Regular Customers (Khách vãng lai)")
)

# Lưu bảng CRM vào Unity Catalog (Giống logic lưu các bảng gold khác của bạn)
save_uc_table(df_crm_final, "dim_crm_analytics")
print("✅ Đã lưu bảng dim_crm_analytics vào Gold Layer.")

In [ ]:
display(spark.table(f"{catalog_name}.{gold_schema}.dim_crm_analytics"))